# Simulation Inspection

Interactive inspection of the initialization, solver, and diagnostics.

In [ ]:
import os
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "6"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "False"
sys.path.append("..")

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from gyaradax import (
    load_geometry,
    get_integrals,
    gksolve,
    gkstep_single,
    init_f,
    load_gkw_k_dump,
    default_state,
    GKParams,
)
from gyaradax.plot_utils import plot_nd, plot_spectra, plot_mode_growth

jax.config.update("jax_enable_x64", True)

BASE = "/restricteddata/ukaea/gyrokinetics/raw/iteration_13_Lin"
geom = load_geometry(BASE)

print("Loaded geometry from:", BASE)
print(
    "Grid sizes (vpar, mu, s, kx, ky):",
    len(geom["intvp"]),
    len(geom["intmu"]),
    len(geom["ints"]),
    len(geom["kxrh"]),
    len(geom["krho"]),
)
print("ixzero, iyzero:", int(geom["ixzero"]), int(geom["iyzero"]))
print(
    "kx range:",
    float(np.min(np.asarray(geom["kxrh"]))),
    float(np.max(np.asarray(geom["kxrh"]))),
)
print(
    "ky range:",
    float(np.min(np.asarray(geom["krho"]))),
    float(np.max(np.asarray(geom["krho"]))),
)

## 1. Initialization (`init_f`)

In [ ]:
df0_raw = init_f(
    geom, amp_init_real=1.0e-4, amp_init_imag=0.0, normalize_per_toroidal_mode=False
)
df0 = init_f(
    geom, amp_init_real=1.0e-4, amp_init_imag=0.0, normalize_per_toroidal_mode=True
)

iyzero = int(geom["iyzero"])
print("df0_raw shape:", df0_raw.shape, "dtype:", df0_raw.dtype)
print("df0 (startup normalized) shape:", df0.shape)
print("max |df0_raw[..., ky=0]|:", float(jnp.max(jnp.abs(df0_raw[..., iyzero]))))
print("max |df0[..., ky=0]|:", float(jnp.max(jnp.abs(df0[..., iyzero]))))

# Verify startup mode normalization (non-zonal ky expected near 1).
phi0_raw, _ = get_integrals(df0_raw, geom)
phi0, _ = get_integrals(df0, geom)
ds = float(np.asarray(geom["ints"])[0])
amp_raw = np.sqrt(ds * jnp.sum(jnp.abs(phi0_raw) ** 2, axis=(0, 1)))
amp_norm = np.sqrt(ds * jnp.sum(jnp.abs(phi0) ** 2, axis=(0, 1)))
print(
    "non-zonal min/max amp after startup normalization:",
    float(np.min(amp_norm[np.arange(len(amp_norm)) != iyzero])),
    float(np.max(amp_norm[np.arange(len(amp_norm)) != iyzero])),
)

In [ ]:
fig = plot_nd(np.abs(np.asarray(df0)))

## 2. Multi-step Execution

In [ ]:
mode_label = np.asarray(geom["mode_label"])
ixzero = int(geom["ixzero"])
iyzero = int(geom["iyzero"])


def run_small_steps(df_init, nsteps=80, params=None):
    if params is None:
        params = GKParams(dt=0.01, naverage=40, disp_par=1.0, disp_x=0.1, disp_y=0.1)
    state = default_state(nky=len(geom["krho"]))

    # JIT the multi-step gksolve which uses jax.lax.scan internally
    solver_jit = jax.jit(gksolve, static_argnums=(4,))

    t0 = time.time()
    df, (phi, fluxes), state = solver_jit(df_init, geom, params, state, nsteps)
    print(f"Executed {nsteps} steps in {time.time()-t0:.4f}s (including JIT)")

    return df, phi, state

## 3. Quick Evolution & Spectra

In [ ]:
params = GKParams(dt=0.01, naverage=40, disp_par=1.0, disp_x=0.1, disp_y=0.1)
nsteps = 80

df80, phi80, state80 = run_small_steps(df0, nsteps=nsteps, params=params)
print(
    "state.step:",
    int(np.asarray(state80.step)),
    "state.time:",
    float(np.asarray(state80.time)),
    "last_growth_rate:",
    float(jnp.mean(np.asarray(state80.last_growth_rate))),
)

kx = np.asarray(geom["kxrh"])
ky = np.asarray(geom["krho"])
fig_spectra = plot_spectra(kx, ky, phi80, title="Spectra after 80 steps")
plt.show()

## 4. Diagnostic History

In [ ]:
# Demonstrate Mode Growth (requires a manual history loop or scan)
def run_history(df_init, nsteps=40):
    params = GKParams(dt=0.01)
    state = default_state(nky=len(geom["krho"]))

    def _step_fn(carry, _):
        d, s = carry
        # Use gkstep_single for manual history tracking
        dn, (pn, _), sn = gkstep_single(d, geom, params, s)
        return (dn, sn), pn

    (_, _), phi_hist = jax.lax.scan(_step_fn, (df_init, state), None, length=nsteps)
    return phi_hist


ky_vals = np.asarray(geom["krho"])
phi_history = np.asarray(run_history(df0, nsteps=40))
t_history = 0.01 * np.arange(40)
ds = float(np.asarray(geom["ints"])[0])
fig_growth = plot_mode_growth(
    t_history, phi_history, ky_indices=[1, 4, 8], ky_values=ky_vals, ds=ds
)
plt.show()

## 5. Linear Checkpoint Evaluation

In [ ]:
res = (
    len(geom["intvp"]),
    len(geom["intmu"]),
    len(geom["ints"]),
    len(geom["kxrh"]),
    len(geom["krho"]),
)
dm2 = load_gkw_k_dump(f"{BASE}/DM2", res)
fds = load_gkw_k_dump(f"{BASE}/FDS", res)

# DM2 at t=319.2, FDS at t=320.0 -> 80 small steps (dt=0.01)
t0 = time.time()
dm2_to_fds_pred, _, _ = run_small_steps(dm2, nsteps=80, params=params)
print(f"DM2->FDS prediction runtime: {time.time()-t0:.2f}s")


def rel_l2(a, b, eps=1e-30):
    na = float(jnp.linalg.norm(a - b))
    nb = float(jnp.linalg.norm(b))
    return na / max(nb, eps)


phi_pred, flux_pred = get_integrals(dm2_to_fds_pred, geom)
phi_fds, flux_fds = get_integrals(fds, geom)

print("Relative L2(df_pred, FDS):", rel_l2(dm2_to_fds_pred, fds))
print("Relative L2(phi_pred, phi_FDS):", rel_l2(phi_pred, phi_fds))

p_pred, e_pred, v_pred = [float(np.asarray(x)) for x in flux_pred]
p_ref, e_ref, v_ref = [float(np.asarray(x)) for x in flux_fds]
print("Fluxes pred (p,e,v):", p_pred, e_pred, v_pred)
print("Fluxes FDS  (p,e,v):", p_ref, e_ref, v_ref)
print(
    "Relative flux errors:",
    abs(p_pred - p_ref) / (abs(p_ref) + 1e-30),
    abs(e_pred - e_ref) / (abs(e_ref) + 1e-30),
    abs(v_pred - v_ref) / (abs(v_ref) + 1e-30),
)

In [ ]:
fig = plot_nd(np.abs(np.asarray(dm2_to_fds_pred)), title="DM2 -> FDS Prediction")

## 6. Nonlinear Smoke Validation

In [ ]:
NONLIN_BASE = "/restricteddata/ukaea/gyrokinetics/raw/iteration_13"
geom_nl = load_geometry(NONLIN_BASE)


def read_dump_time(dat_path):
    with open(dat_path, "r", encoding="utf-8") as f:
        text = f.read()
    import re

    m = re.search(r"TIME\s*=\s*([0-9eE+\-.]+)", text)
    if m is None:
        raise ValueError(f"TIME entry not found in {dat_path}")
    return float(m.group(1))


def selected_ky_representatives(iyzero, nky):
    candidates = [1, nky // 2, nky - 1]
    out = []
    for ky in candidates:
        ky = int(np.clip(ky, 0, nky - 1))
        if ky == iyzero:
            continue
        if ky not in out:
            out.append(ky)
    if not out:
        out = [int((iyzero + 1) % nky)]
    return out


def run_small_steps_on_geom(df_init, geometry, params, nsteps, state=None):
    if state is None:
        state = default_state(nky=len(geom["krho"]))

    def _gkstep_wrap(carry, _):
        d, s = carry
        dn, (pn, fl), sn = gkstep_single(d, geometry, params, s)
        return (dn, sn), (pn, fl[1], sn.time, sn.last_growth_rate)

    t0 = time.time()
    (df, state), (phi_hist, eflux_hist, time_hist, growth_hist) = jax.jit(
        lambda d0, s0: jax.lax.scan(_gkstep_wrap, (d0, s0), None, length=nsteps)
    )(df_init, state)
    print(
        f"Executed {nsteps} steps with history in {time.time()-t0:.4f}s (including JIT)"
    )

    return (
        df,
        np.asarray(phi_hist),
        np.asarray(eflux_hist),
        np.asarray(time_hist),
        np.asarray(growth_hist),
        state,
    )

In [ ]:
res_nl = (
    len(geom_nl["intvp"]),
    len(geom_nl["intmu"]),
    len(geom_nl["ints"]),
    len(geom_nl["kxrh"]),
    len(geom_nl["krho"]),
)
start_name = "100"
end_name = "101"

start_df = load_gkw_k_dump(f"{NONLIN_BASE}/{start_name}", res_nl)
end_df_ref = load_gkw_k_dump(f"{NONLIN_BASE}/{end_name}", res_nl)

t_start = read_dump_time(f"{NONLIN_BASE}/{start_name}.dat")
t_end = read_dump_time(f"{NONLIN_BASE}/{end_name}.dat")
nsteps = int(round((t_end - t_start) / 0.01))

params_nl = GKParams(
    dt=0.01,
    naverage=40,
    non_linear=True,
)

state_nl0 = type(default_state())(
    time=jnp.array(t_start, dtype=jnp.float64),
    step=jnp.array(0, dtype=jnp.int32),
    accumulated_norm_factor=jnp.ones(len(geom_nl["krho"]), dtype=jnp.float64),
    window_start_amp=jnp.ones(len(geom_nl["krho"]), dtype=jnp.float64),
    last_growth_rate=jnp.zeros(len(geom_nl["krho"]), dtype=jnp.float64),
)

pred_df_nl, phi_hist_nl, eflux_hist_nl, time_hist_nl, growth_hist_nl, state_nl = (
    run_small_steps_on_geom(
        start_df,
        geom_nl,
        params_nl,
        nsteps=nsteps,
        state=state_nl0,
    )
)

subset_rel_l2_nl = rel_l2(pred_df_nl, end_df_ref)
print("Nonlinear smoke subset rel_l2:", subset_rel_l2_nl)